# Fuel Data Cleaning

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 配置matplotlib中文显示
import platform
from rapidfuzz import fuzz, process
from sklearn.cluster import AgglomerativeClustering
system_name = platform.system()

if system_name == 'Windows':
    # Windows系统
    plt.rcParams['font.family'] = ['SimHei']
elif system_name == 'Darwin':
    # macOS系统
    plt.rcParams['font.family'] = ['PingFang HK']
elif system_name == 'Linux':
    # Linux系统（可能需要根据具体发行版进行调整）
    plt.rcParams['font.family'] = ['DejaVu Sans']
else:
    # 其他系统或无法识别系统，默认字体
    plt.rcParams['font.family'] = ['sans-serif']
plt.rcParams['axes.unicode_minus'] = False  # 用来正常显示负号

In [2]:
# import data
file_path = 'Resources/Fuel_合并.xlsx'
xlsx = pd.ExcelFile(file_path)
print(xlsx.sheet_names)

['油耗信息', '设备信息']


# 处理油耗信息

In [41]:
df_sheet1 = pd.read_excel(file_path, sheet_name=0)

In [42]:
print(df_sheet1.shape)
print(df_sheet1.head())

(222752, 6)
          日期     班次            设备名称    设备编号   油品种类 油品消耗
0 2019-08-04  Night  TEREX TR100#56  HT0056  IC IC  476
1 2019-08-04  Night  TEREX TR100#58  HT0058  IC IC  462
2 2019-08-04  Night  TEREX TR100#59  HT0059  IC IC  451
3 2019-08-04  Night  TEREX TR100#60  HT0060  IC IC  454
4 2019-08-04  Night  TEREX TR100#61  HT0061  IC IC  502


In [55]:
# 转换格式为浮点数
df_sheet1['油品消耗'] = df_sheet1['油品消耗'].astype(str).str.strip()
df_sheet1['油品消耗'] = pd.to_numeric(df_sheet1['油品消耗'], errors='coerce')
df_sheet1.dropna(subset=['油品消耗'],inplace=True)

In [56]:
# 删除NaN值和零值
print(f"NaN counts:{df_sheet1['油品消耗'].isnull().sum()}")
print(f"Zero counts:{(df_sheet1['油品消耗']==0).sum()}")

# 先处理零值再处理NaN值
df_sheet1 = df_sheet1[df_sheet1['油品消耗']!=0]
df_sheet1 = df_sheet1[~df_sheet1['油品消耗'].isnull()]

NaN counts:0
Zero counts:8689


## 格式化名称和编号

In [58]:
# Format name and number
df_sheet1['设备名称'] = df_sheet1['设备名称'].str.strip()
df_sheet1['设备编号'] = df_sheet1['设备编号'].str.strip()
df_sheet1

,日期,班次,设备名称,设备编号,油品种类,油品消耗
0,2019-08-04,Night,TEREX TR100#56,HT0056,IC IC,476.00
1,2019-08-04,Night,TEREX TR100#58,HT0058,IC IC,462.00
2,2019-08-04,Night,TEREX TR100#59,HT0059,IC IC,451.00
3,2019-08-04,Night,TEREX TR100#60,HT0060,IC IC,454.00
4,2019-08-04,Night,TEREX TR100#61,HT0061,IC IC,502.00
...,...,...,...,...,...,...
222747,2026-04-30,Night,TOYOTA LANDCRUISER78 LV#1225,LV1225,Золотой /300007/,60.00
222748,2026-04-30,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,LV1226,Золотой /300007/,30.84
222749,2026-04-30,Night,North Benz ST#137 /ST0137/,ST0137,Золотой /300007/,173.35
222750,2026-04-30,Night,North Benz ST#141,ST141,Золотой /300007/,350.00


In [59]:
# Find duplicates
print(df_sheet1.duplicated().sum())
print(df_sheet1.shape)
df_sheet1 = df_sheet1.drop_duplicates()
print(df_sheet1.shape)

5424
(214060, 6)
(208636, 6)


In [60]:
# Find null values
print(df_sheet1['设备名称'].isnull().sum())
col_to_drop = df_sheet1[df_sheet1['设备名称'].isnull()&(df_sheet1['设备编号'].isnull())]
print(col_to_drop)
df_sheet1 = df_sheet1.drop(col_to_drop.index)
print(df_sheet1.shape)

13
             日期   班次 设备名称 设备编号     油品种类   油品消耗
5137 2019-12-20  Day  NaN  NaN  Primary   98.0
5138 2019-12-20  Day  NaN  NaN  Primary  100.0
5197 2019-12-21  Day  NaN  NaN  Primary  219.0
5198 2019-12-21  Day  NaN  NaN  Primary  237.0
5199 2019-12-21  Day  NaN  NaN  Primary  145.0
5200 2019-12-21  Day  NaN  NaN  Primary  193.0
(208630, 6)


In [61]:
df_sheet1[df_sheet1.isnull().any(axis=1)]

,日期,班次,设备名称,设备编号,油品种类,油品消耗
10,2019-08-13,Day,GREIDER,NaN,Pimary Energy,310.0
5483,2021-01-01,Day,TEREX TR100 #1056,NaN,НИК,500.5
5484,2021-01-01,Day,TEREX TR100 #1057,NaN,НИК,382.5
5485,2021-01-01,Day,TEREX TR100 #1058,NaN,НИК,453.0
5486,2021-01-01,Day,TEREX TR100 #1059,NaN,НИК,545.0
...,...,...,...,...,...,...
170774,2025-04-26,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,NaN,Primary /300003/,28.0
171144,2025-04-27,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,NaN,Шунхлай /300008/,24.0
171498,2025-04-28,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,NaN,Шунхлай /300008/,36.0
171854,2025-04-29,Night,9301УБР/ LV#1226 /TOYOTA /LAND 78/,NaN,Шунхлай /300008/,27.0


## 聚类

In [71]:
names = df_sheet1['设备名称'].unique()

# 构建相似度矩阵
n = len(names)
matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):
        matrix[i, j] = fuzz.ratio(names[i], names[j])

# 转为距离（越小越相似）
distance = 100 - matrix

clustering = AgglomerativeClustering(
    linkage='average',
    distance_threshold=20,
    n_clusters=None
)

labels = clustering.fit_predict(distance)
mapping = dict(zip(names, labels))

In [73]:
df_sheet1['cluster'] = df_sheet1['设备名称'].map(mapping)

In [80]:
len(names)

441

In [87]:
import pandas as pd
import difflib

# 假设你的原始数据已经加载为 df_sheet1
# df_sheet1 = pd.read_excel('你的原始文件.xlsx')

def cluster_device_names(df, similarity_threshold=0.65):
    # 1. 获取所有不重复的且非空的设备名称
    unique_names = df['设备名称'].dropna().unique().tolist()

    # 记录每个名称对应的当前设备编号（如果有多个，取第一个作为参考）
    name_to_id = df.dropna(subset=['设备编号']).drop_duplicates(subset=['设备名称']).set_index('设备名称')['设备编号'].to_dict()

    clusters = []
    visited = set()

    # 2. 遍历并寻找相似名称进行聚类
    cluster_id = 1
    for name in unique_names:
        if name in visited:
            continue

        # 使用 difflib 获取相似度高于阈值的名称
        matches = difflib.get_close_matches(name, unique_names, n=80, cutoff=similarity_threshold)

        # 确定这一组的“标准名称”和“标准编号”
        # 优先使用字数较短的作为标准名称，或者如果有已经存在编号的，优先用它的编号
        standard_name = matches[0]
        suggested_id = None
        for m in matches:
            if m in name_to_id:
                suggested_id = name_to_id[m]
                standard_name = m
                break

        # 如果都没有编号，生成一个临时的建议编号
        if suggested_id is None:
            suggested_id = f"TEMP_ID_{cluster_id:04d}"

        # 将结果存入列表
        for m in matches:
            clusters.append({
                '原始设备名称': m,
                '当前设备编号': name_to_id.get(m, ''),
                '算法推荐标准名称': standard_name,
                '算法推荐设备编号': suggested_id,
                '人工确认设备编号': suggested_id  # 留给人工修改的列，默认填入推荐值
            })
            visited.add(m)

        cluster_id += 1

    # 处理那些孤立的、没有被匹配到的名称
    for name in unique_names:
        if name not in visited:
            clusters.append({
                '原始设备名称': name,
                '当前设备编号': name_to_id.get(name, ''),
                '算法推荐标准名称': name,
                '算法推荐设备编号': name_to_id.get(name, f"TEMP_ID_{cluster_id:04d}"),
                '人工确认设备编号': name_to_id.get(name, f"TEMP_ID_{cluster_id:04d}")
            })
            cluster_id += 1

    return pd.DataFrame(clusters)

# 执行聚类
df_mapping = cluster_device_names(df_sheet1, similarity_threshold=0.65)

# 按照推荐编号排序，方便人工把同一类的名称放到一起看
df_mapping = df_mapping.sort_values('算法推荐设备编号')

# 导出为 Excel 供人工检查
output_file = '设备名称匹配确认表_待人工检查.xlsx'
df_mapping.to_excel(output_file, index=False)
print(f"聚类匹配结果已导出至：{output_file}，请打开检查并修改【人工确认设备编号】列。")

聚类匹配结果已导出至：设备名称匹配确认表_待人工检查.xlsx，请打开检查并修改【人工确认设备编号】列。


In [90]:
import pandas as pd
import re

# 1. 配置基础信息
STANDARD_PREFIXES = [
    'CHU', 'CP', 'CR', 'DZ', 'EX', 'FL', 'GE', 'GM', 'GR', 'HE',
    'HT', 'LO', 'LP', 'LV', 'ME', 'ST', 'SV', 'WD', 'WP', 'WT', 'XD'
]

df = df_sheet1.copu()

def extract_standard_id(text, prefixes):
    """
    从文本中提取标准编号逻辑
    格式：前缀 + (可选#) + 数字 -> 前缀#0000
    """
    if pd.isna(text):
        return None

    text = str(text).upper()
    # 构建正则表达式：匹配前缀 + 任意非数字字符(如#或空格) + 数字
    # 例如: (DZ|ST|...)[^\d]*(\d+)
    prefix_pattern = f"({'|'.join(prefixes)})"
    regex = prefix_pattern + r"[^\d]*(\d+)"

    match = re.search(regex, text)
    if match:
        prefix = match.group(1)
        number = match.group(2).zfill(4) # 补全为4位数字
        return f"{prefix}#{number}"
    return None

def process_equipment_data(df, prefixes):
    # 步骤1：尝试从“设备编号”提取
    df['从编号提取'] = df['设备编号'].apply(lambda x: extract_standard_id(x, prefixes))

    # 步骤2：如果从编号提取失败，尝试从“设备名称”提取
    df['从名称提取'] = df['设备名称'].apply(lambda x: extract_standard_id(x, prefixes))

    # 步骤3：确定最终标准编号
    # 逻辑：优先用编号提取的结果，如果没有则用名称提取的结果
    df['最终标准编号'] = df['从编号提取'].combine_first(df['从名称提取'])

    # 步骤4：标记匹配状态
    def check_status(row):
        if pd.notna(row['最终标准编号']):
            return 'Match_Success'
        else:
            # 检查是否包含数字但前缀不对
            if re.search(r'\d+', str(row['设备编号'])) or re.search(r'\d+', str(row['设备名称'])):
                return 'Prefix_Mismatch_Or_Missing'
            return 'No_Data'

    df['匹配状态'] = df.apply(check_status, axis=1)

    # 步骤5：按照最终标准编号排序，方便核对同一设备
    df = df.sort_values(by=['最终标准编号', '设备名称'], na_position='last')

    return df[['设备名称', '设备编号', '最终标准编号', '匹配状态']].drop_duplicates()

# 执行处理
result_df = process_equipment_data(df_sheet1, STANDARD_PREFIXES)

# 导出结果
output_file = '设备编号核对清单_精准匹配版.xlsx'
result_df.to_excel(output_file, index=False)

print(f"处理完成！结果已保存至: {output_file}")
print(result_df[['设备名称', '设备编号', '最终标准编号', '匹配状态']].head(10))

处理完成！结果已保存至: 设备编号核对清单_精准匹配版.xlsx
                            设备名称               设备编号    最终标准编号           匹配状态
750             CHUU-COMPERESSOR            CHUU001  CHU#0001  Match_Success
838                      CP-0006            CHUU001  CHU#0001  Match_Success
1034           ATLASCOPCO CP#006              CP006   CP#0006  Match_Success
17832           COMPRESSORS #006             CP0006   CP#0006  Match_Success
2562   COMPRESSORS #006 /CP0005/             CP0006   CP#0006  Match_Success
5545   CPC30-WS1 /Сэрээт өргөгч/                NaN   CP#0030  Match_Success
27730  CPC30-WS1 /Сэрээт өргөгч/               #133   CP#0030  Match_Success
31515  CPC30-WS1 /Сэрээт өргөгч/  CPC30 #133 /#133/   CP#0030  Match_Success
38077        CPC100  #134 /#134/               #134   CP#0100  Match_Success
5544   CPC100-W5 /Сэрээт өргөгч/                NaN   CP#0100  Match_Success
